In [ ]:
%load_ext autoreload
%autoreload 2

import holidays
import httpx
import pandas as pd

In [ ]:
import os

ML_ENGINE_BASE_URL = os.environ.get("ML_ENGINE_BASE_URL", "http://localhost:8000").rstrip("/")
REALIZED_PATH = "/api/v1/energy/consumption/realized"
url = f"{ML_ENGINE_BASE_URL}{REALIZED_PATH}"
params = {
    "start": "2024-01-01",
    "end": "2024-01-10",
}
pd.set_option("display.max_rows", 100)

response = httpx.get(url, params=params, timeout=30.0)
response.raise_for_status()
data = response.json()
print(data)

In [ ]:
# Transform the data to TimeSeries

# Transform the data to dataframe
df = pd.DataFrame(data)

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

# set the timestamp as index
df.set_index("timestamp", inplace=True)

df.head()

In [ ]:
# Resampling the data to hourly (mean of the megawatts values)

df_resampled = df.resample("1h").mean()

df_resampled.head()

In [ ]:
# Feature engineering
# We will create new features (columns) bases on our actual data to improve the model performance

# 1. Day of the week
df_resampled["day_of_week"] = df_resampled.index.dayofweek
df_resampled["hour_of_day"] = df_resampled.index.hour
df_resampled["month"] = df_resampled.index.month

df_resampled["is_weekend"] = df_resampled.index.dayofweek.isin([5, 6]).astype(int)

# bank holidays
years_present = df_resampled.index.year.unique().tolist()
fr_bank_holidays = holidays.France(years=years_present)

# Holiday lookup uses calendar dates; compare via .index.date, not raw timestamps.
df_resampled["is_bank_holiday"] = (
    pd.Index(df_resampled.index.date).isin(fr_bank_holidays).astype(int)
)

df_resampled.head()

In [ ]:
# Suppress the empty data
df_resampled.isna().sum()

# Interpolate NaNs linearly (keeps continuity; dropping would leave gaps).
df_resampled = df_resampled.interpolate(method="linear")

# Drop edge NaNs that interpolation cannot fill (series start/end).
df_resampled = df_resampled.dropna()
df_resampled.head()

# Export CSV; index is the timestamp and is written as a column when needed.
df_resampled.to_csv("../data/processed/energy_consumption_realized_resampled.csv", index=True)